In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

df = pd.read_csv("cleaned_data.csv")

print("Dataset Shape:", df.shape)

df.head()



In [ ]:
print(df.columns)

In [ ]:
X = df.drop(columns=["loan_status"])

y_clf = df["loan_status"]

print("Feature Shape:", X.shape)
print("Target Shape:", y_clf.shape)

print("\nTarget Distribution:")
print(y_clf.value_counts())

In [ ]:
categorical_columns = X.select_dtypes(
    include=["object", "category"]
).columns

print("Categorical Columns:")
print(categorical_columns)

In [ ]:
loan_grade_mapping = {
    "A": 1,
    "B": 2,
    "C": 3,
    "D": 4,
    "E": 5,
    "F": 6,
    "G": 7
}

X["loan_grade"] = X["loan_grade"].map(
    loan_grade_mapping
)
categorical_columns = X.select_dtypes(
    include=["object", "category"]
).columns

X = pd.get_dummies(
    X,
    columns=categorical_columns,
    drop_first=True,
    dtype=int
)

print("Encoded Feature Shape:", X.shape)

X.head()

In [ ]:
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X,
    y_clf,
    test_size=0.2,
    random_state=42
)

print("X Train Shape:", X_train_clf.shape)
print("X Test Shape :", X_test_clf.shape)

print("Y Train Shape:", y_train_clf.shape)
print("Y Test Shape :", y_test_clf.shape)
print(y_test_clf.value_counts())

In [ ]:
scaler = StandardScaler()

scaler.fit(X_train_clf)

X_train_clf_scaled = scaler.transform(
    X_train_clf
)

X_test_clf_scaled = scaler.transform(
    X_test_clf
)

print("Scaling Completed")

print(
    "Training Shape:",
    X_train_clf_scaled.shape
)

print(
    "Test Shape:",
    X_test_clf_scaled.shape
)

In [ ]:
# ============================================================
# TASK 1 - DECISION TREE BASELINE
# ============================================================

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score


# ------------------------------------------------------------
# CREATE MODEL
# ------------------------------------------------------------

decision_tree = DecisionTreeClassifier(
    random_state=42
)


# ------------------------------------------------------------
# TRAIN MODEL
# ------------------------------------------------------------

decision_tree.fit(
    X_train_clf_scaled,
    y_train_clf
)


# ------------------------------------------------------------
# PREDICTIONS
# ------------------------------------------------------------

y_train_pred_dt = decision_tree.predict(
    X_train_clf_scaled
)

y_test_pred_dt = decision_tree.predict(
    X_test_clf_scaled
)


# ------------------------------------------------------------
# ACCURACY
# ------------------------------------------------------------

train_accuracy_dt = accuracy_score(
    y_train_clf,
    y_train_pred_dt
)

test_accuracy_dt = accuracy_score(
    y_test_clf,
    y_test_pred_dt
)


# ------------------------------------------------------------
# ACCURACY GAP
# ------------------------------------------------------------

accuracy_gap = train_accuracy_dt - test_accuracy_dt


# ------------------------------------------------------------
# RESULTS
# ------------------------------------------------------------

print("=" * 60)
print("DECISION TREE BASELINE PERFORMANCE")
print("=" * 60)

print(f"Training Accuracy : {train_accuracy_dt:.4f}")
print(f"Test Accuracy     : {test_accuracy_dt:.4f}")
print(f"Accuracy Gap      : {accuracy_gap:.4f}")

In [ ]:
Decision Tree Baseline

An unconstrained `DecisionTreeClassifier` was trained using the default `max_depth=None`. The model was evaluated on both the training and test datasets to assess its generalization performance and identify signs of overfitting.

Model Performance

| Metric                  |      Accuracy |
| ----------------------- | ------------: |
| Training Accuracy       | 1.0000 (100%) |
| Test Accuracy           |  0.8900 (89%) |
| Train-Test Accuracy Gap |  0.1100 (11%) |

Overfitting Analysis

The Decision Tree achieved a training accuracy of **100%**, indicating that it perfectly classified all observations in the training dataset. However, its test accuracy decreased to **89%**, producing an accuracy gap of **11 percentage points**.

This difference indicates that the unconstrained Decision Tree shows clear signs of **overfitting**. Since `max_depth=None`, the tree is allowed to grow deeply and create highly specific decision rules. As a result, the model may learn noise and patterns that are unique to the training dataset rather than only learning relationships that generalize to unseen data.

Although a test accuracy of 89% is relatively strong, the perfect training accuracy combined with the 11% train-test gap suggests that the model has high variance and does not generalize as consistently as its training performance suggests.

Why Decision Trees Are High-Variance Models

Decision Trees are considered high-variance models because they construct their structure greedily. At each node, the algorithm selects the split that provides the best immediate improvement according to the splitting criterion. Once a split is selected, the tree continues growing without revisiting and correcting earlier splitting decisions.

Small changes in the training dataset can therefore produce different early splits and lead to substantially different tree structures and predictions. An unconstrained tree can also continue creating branches until it captures highly specific training observations.

Conclusion

The baseline Decision Tree demonstrates overfitting because it achieved **100% training accuracy but only 89% test accuracy**, resulting in an **11% accuracy gap**. This confirms the high-variance nature of an unconstrained Decision Tree.

To improve robustness and reduce variance, ensemble techniques such as **Bagging** and **Random Forest** will be evaluated next. These methods combine multiple Decision Trees to produce more stable predictions and improve generalization to unseen data.


In [ ]:
# ============================================================
# CONTROLLED DECISION TREE
# ============================================================

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

controlled_tree = DecisionTreeClassifier(
    max_depth=5,
    min_samples_split=20,
    random_state=42
)

print(controlled_tree)

In [ ]:
# ============================================================
# TRAIN CONTROLLED DECISION TREE
# ============================================================

controlled_tree.fit(
    X_train_clf_scaled,
    y_train_clf
)

print("Controlled Decision Tree Training Completed")

In [ ]:
# ============================================================
# PREDICTIONS
# ============================================================

y_train_pred_controlled = controlled_tree.predict(
    X_train_clf_scaled
)

y_test_pred_controlled = controlled_tree.predict(
    X_test_clf_scaled
)

In [ ]:
# ============================================================
# ACCURACY CALCULATION
# ============================================================

train_accuracy_controlled = accuracy_score(
    y_train_clf,
    y_train_pred_controlled
)

test_accuracy_controlled = accuracy_score(
    y_test_clf,
    y_test_pred_controlled
)

controlled_gap = (
    train_accuracy_controlled
    - test_accuracy_controlled
)

print("=" * 60)
print("CONTROLLED DECISION TREE PERFORMANCE")
print("=" * 60)

print(
    f"Training Accuracy : {train_accuracy_controlled:.4f}"
)

print(
    f"Test Accuracy     : {test_accuracy_controlled:.4f}"
)

print(
    f"Accuracy Gap      : {controlled_gap:.4f}"
)

In [ ]:
# ============================================================
# MODEL COMPARISON
# ============================================================

comparison_df = pd.DataFrame({
    "Model": [
        "Unconstrained Decision Tree",
        "Controlled Decision Tree"
    ],
    "Training Accuracy": [
        train_accuracy_dt,
        train_accuracy_controlled
    ],
    "Test Accuracy": [
        test_accuracy_dt,
        test_accuracy_controlled
    ],
    "Accuracy Gap": [
        accuracy_gap,
        controlled_gap
    ]
})

print(comparison_df)

In [ ]:

Controlled Decision Tree

A second Decision Tree classifier was trained using `max_depth=5` and `min_samples_split=20`. These hyperparameters were introduced to control tree complexity and reduce the overfitting observed in the unconstrained Decision Tree.

Model Performance Comparison

| Model                       | Training Accuracy | Test Accuracy | Train-Test Gap |
| --------------------------- | ----------------: | ------------: | -------------: |
| Unconstrained Decision Tree |            1.0000 |        0.8900 |         0.1100 |
| Controlled Decision Tree    |            0.9117 |        0.9126 |        -0.0009 |

Interpretation of the Results

The unconstrained Decision Tree achieved **100% training accuracy** but only **89.00% test accuracy**, resulting in a train-test gap of approximately **11 percentage points**. This large performance gap indicates that the model overfit the training dataset. The tree was sufficiently complex to memorize training-specific patterns and noise, resulting in weaker generalization to unseen data.

In comparison, the controlled Decision Tree achieved **91.17% training accuracy** and **91.26% test accuracy**. The train-test gap was approximately **-0.09 percentage points**, which is substantially smaller than the 11% gap observed in the unconstrained model.

The slightly higher test accuracy than training accuracy is not a concern. The difference is extremely small and can occur because of normal sampling variation between the training and test datasets. It does not indicate overfitting.

Role of `max_depth`

The `max_depth` parameter limits how deep the Decision Tree can grow. Setting `max_depth=5` prevents the model from creating excessively deep branches and highly specific decision rules.

Restricting tree depth reduces model variance because the model is less sensitive to small variations and noise in the training dataset. However, this restriction can introduce additional bias because the model has less flexibility to learn very complex relationships.

Role of `min_samples_split`

The `min_samples_split` parameter specifies the minimum number of samples required for a node to be considered for splitting.

Setting `min_samples_split=20` prevents the model from creating additional splits based on small subsets of observations. Such splits may respond to random noise rather than meaningful patterns. Therefore, this parameter helps create a more stable and generalizable tree.

Comparison with the Unconstrained Tree

The train-test accuracy gap decreased from approximately **11% to almost 0%** after controlling the complexity of the Decision Tree. In addition, test accuracy improved from **89.00% to 91.26%**.

Although training accuracy decreased from **100% to 91.17%**, this reduction is beneficial because the model is no longer memorizing the training dataset. Instead, the controlled tree appears to capture more general patterns that transfer effectively to unseen data.

This demonstrates the **bias-variance trade-off**. The unconstrained Decision Tree had low training bias but high variance. Restricting the tree's complexity introduced some bias but substantially reduced variance and improved generalization.

Conclusion

The controlled Decision Tree performs better than the unconstrained Decision Tree in terms of generalization. Using `max_depth=5` and `min_samples_split=20` reduced the train-test accuracy gap from **11% to approximately 0%** while improving test accuracy from **89.00% to 91.26%**.

Therefore, the controlled Decision Tree is more robust and less prone to overfitting than the unconstrained baseline model. These results demonstrate that controlling model complexity can improve performance on unseen data even when training accuracy decreases.




In [ ]:
# ============================================================
# TASK - GINI VS ENTROPY COMPARISON
# ============================================================

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
import pandas as pd


# ------------------------------------------------------------
# GINI DECISION TREE
# ------------------------------------------------------------

gini_tree = DecisionTreeClassifier(
    criterion="gini",
    max_depth=5,
    random_state=42
)

gini_tree.fit(
    X_train_clf_scaled,
    y_train_clf
)

y_pred_gini = gini_tree.predict(
    X_test_clf_scaled
)

gini_accuracy = accuracy_score(
    y_test_clf,
    y_pred_gini
)


# ------------------------------------------------------------
# ENTROPY DECISION TREE
# ------------------------------------------------------------

entropy_tree = DecisionTreeClassifier(
    criterion="entropy",
    max_depth=5,
    random_state=42
)

entropy_tree.fit(
    X_train_clf_scaled,
    y_train_clf
)

y_pred_entropy = entropy_tree.predict(
    X_test_clf_scaled
)

entropy_accuracy = accuracy_score(
    y_test_clf,
    y_pred_entropy
)


# ------------------------------------------------------------
# COMPARISON TABLE
# ------------------------------------------------------------

criterion_comparison = pd.DataFrame({
    "Criterion": [
        "Gini",
        "Entropy"
    ],
    "Test Accuracy": [
        gini_accuracy,
        entropy_accuracy
    ]
})


print("=" * 60)
print("GINI VS ENTROPY COMPARISON")
print("=" * 60)

print(criterion_comparison)

print("\nGini Test Accuracy    :", round(gini_accuracy, 4))
print("Entropy Test Accuracy :", round(entropy_accuracy, 4))

In [ ]:
Gini vs Entropy Comparison

Two Decision Tree classifiers were trained using max_depth=5 under the same training and testing conditions. The first model used criterion='gini', while the second model used criterion='entropy'.

Model Performance
Splitting Criterion	Test Accuracy
Gini	0.9126 (91.26%)
Entropy	0.9132 (91.32%)


Gini Impurity

Gini impurity measures the level of class mixing within a Decision Tree node.

The formula for Gini impurity is:

Gini = 1 − Σ pᵢ²

where pᵢ represents the proportion of samples belonging to class i within the node.

A node with Gini = 0 is a completely pure node. This means that all samples in the node belong to the same class. Therefore, there is no class mixture within the node.

Entropy

Entropy measures the uncertainty or disorder in the class distribution of a node.

The formula for Entropy is:

Entropy = −Σ pᵢ log₂(pᵢ)

where pᵢ represents the proportion of samples belonging to class i.

A lower entropy value represents a purer node, while a higher entropy value indicates greater class uncertainty. An entropy value of zero means that all samples in the node belong to one class.

Performance Interpretation

The Gini-based Decision Tree achieved a test accuracy of 91.26%, while the Entropy-based Decision Tree achieved a test accuracy of 91.32%.

Entropy produced the slightly higher test accuracy. However, the difference between the two models is only approximately 0.06 percentage points.

This difference is extremely small and does not provide strong evidence that Entropy is meaningfully better than Gini for this dataset. Both splitting criteria appear to identify very similar predictive patterns in the credit-risk data.

The result suggests that the choice between Gini impurity and Entropy has little practical impact on the predictive accuracy of the controlled Decision Tree for this problem.

Conclusion

The Entropy Decision Tree achieved the highest test accuracy of 91.32%, compared with 91.26% for Gini. Although Entropy technically produced the better result, the improvement is negligible.

Therefore, both criteria can be considered to have comparable performance on this dataset. The choice of splitting criterion is not a major factor affecting model performance in this experiment.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

# Create Random Forest model
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42
)

# Train Random Forest
rf_model.fit(
    X_train_clf_scaled,
    y_train_clf
)

# Predictions
y_pred_rf = rf_model.predict(X_test_clf_scaled)

y_proba_rf = rf_model.predict_proba(
    X_test_clf_scaled
)[:, 1]

# Performance
rf_train_accuracy = accuracy_score(
    y_train_clf,
    rf_model.predict(X_train_clf_scaled)
)

rf_test_accuracy = accuracy_score(
    y_test_clf,
    y_pred_rf
)

rf_auc = roc_auc_score(
    y_test_clf,
    y_proba_rf
)

print("=" * 60)
print("RANDOM FOREST PERFORMANCE")
print("=" * 60)

print(f"Training Accuracy : {rf_train_accuracy:.4f}")
print(f"Test Accuracy     : {rf_test_accuracy:.4f}")
print(f"ROC-AUC           : {rf_auc:.4f}")

In [ ]:
print(rf_model)

In [ ]:
import pandas as pd

feature_importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf_model.feature_importances_
})

feature_importance_df = (
    feature_importance_df
    .sort_values(
        by="Importance",
        ascending=False
    )
    .reset_index(drop=True)
)

print("=" * 60)
print("TOP 5 IMPORTANT FEATURES")
print("=" * 60)

print(feature_importance_df.head(5))

In [ ]:
print("Training features:",
      X_train_clf_scaled.shape[1])

print("Random Forest importances:",
      len(rf_model.feature_importances_))

print("X column names:",
      len(X.columns))

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
# ============================================================
# GRADIENT BOOSTING CLASSIFIER
# ============================================================

gb_model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

gb_model.fit(
    X_train_clf_scaled,
    y_train_clf
)
# Predictions
y_pred_gb = gb_model.predict(
    X_test_clf_scaled
)

y_proba_gb = gb_model.predict_proba(
    X_test_clf_scaled
)[:, 1]


# Accuracy
gb_train_accuracy = accuracy_score(
    y_train_clf,
    gb_model.predict(X_train_clf_scaled)
)

gb_test_accuracy = accuracy_score(
    y_test_clf,
    y_pred_gb
)

# ROC-AUC
gb_auc = roc_auc_score(
    y_test_clf,
    y_proba_gb
)


print("=" * 60)
print("GRADIENT BOOSTING PERFORMANCE")
print("=" * 60)

print(f"Training Accuracy : {gb_train_accuracy:.4f}")
print(f"Test Accuracy     : {gb_test_accuracy:.4f}")
print(f"ROC-AUC           : {gb_auc:.4f}")

In [ ]:
model_comparison = pd.DataFrame({
    "Model": [
        "Random Forest",
        "Gradient Boosting"
    ],
    "Training Accuracy": [
        rf_train_accuracy,
        gb_train_accuracy
    ],
    "Test Accuracy": [
        rf_test_accuracy,
        gb_test_accuracy
    ],
    "ROC-AUC": [
        rf_auc,
        gb_auc
    ]
})

print("=" * 70)
print("MODEL PERFORMANCE COMPARISON")
print("=" * 70)

print(model_comparison)

In [ ]:
# ============================================================
# FIVE LOWEST IMPORTANCE FEATURES
# ============================================================

lowest_5_features = (
    feature_importance_df
    .tail(5)
)

print("=" * 60)
print("5 LOWEST IMPORTANCE FEATURES")
print("=" * 60)

print(lowest_5_features)

lowest_5_feature_names = (
    lowest_5_features["Feature"]
    .tolist()
)

print("\nFeatures to remove:")

for feature in lowest_5_feature_names:
    print(feature)

In [ ]:
X_train_scaled_df = pd.DataFrame(
    X_train_clf_scaled,
    columns=X.columns
)

X_test_scaled_df = pd.DataFrame(
    X_test_clf_scaled,
    columns=X.columns
)

print(
    "Training shape before ablation:",
    X_train_scaled_df.shape
)

print(
    "Test shape before ablation:",
    X_test_scaled_df.shape
)

In [ ]:
X_train_reduced = X_train_scaled_df.drop(
    columns=lowest_5_feature_names
)

X_test_reduced = X_test_scaled_df.drop(
    columns=lowest_5_feature_names
)


print(
    "Training shape after ablation:",
    X_train_reduced.shape
)

print(
    "Test shape after ablation:",
    X_test_reduced.shape
)

In [ ]:
# ============================================================
# REDUCED RANDOM FOREST
# ============================================================

rf_reduced_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42
)

rf_reduced_model.fit(
    X_train_reduced,
    y_train_clf
)

y_proba_rf_reduced = (
    rf_reduced_model
    .predict_proba(X_test_reduced)[:, 1]
)

rf_reduced_auc = roc_auc_score(
    y_test_clf,
    y_proba_rf_reduced
)
ablation_comparison = pd.DataFrame({
    "Model": [
        "Full Random Forest",
        "Reduced Random Forest"
    ],
    "Number of Features": [
        X_train_scaled_df.shape[1],
        X_train_reduced.shape[1]
    ],
    "ROC-AUC": [
        rf_auc,
        rf_reduced_auc
    ]
})

print("=" * 70)
print("FEATURE ABLATION STUDY")
print("=" * 70)

print(ablation_comparison)

In [ ]:
auc_difference = rf_reduced_auc - rf_auc

print("\nFull Model AUC    :", round(rf_auc, 6))
print("Reduced Model AUC :", round(rf_reduced_auc, 6))
print("AUC Difference    :", round(auc_difference, 6))

if auc_difference > 0:

    print(
        "\nInterpretation: Removing the five lowest-importance "
        "features improved ROC-AUC."
    )

    print(
        "The removed features may have introduced noise."
    )

elif abs(auc_difference) < 0.001:

    print(
        "\nInterpretation: ROC-AUC remained almost unchanged."
    )

    print(
        "The removed features appear to provide very little "
        "predictive information."
    )

else:

    print(
        "\nInterpretation: ROC-AUC decreased after feature removal."
    )

    print(
        "The removed features were contributing some "
        "predictive information."
    )

In [ ]:
Random Forest Classification

A Random Forest classifier was trained using `n_estimators=100`, `max_depth=10`, and `random_state=42`.

Random Forest Performance

| Metric            |  Score |
| ----------------- | -----: |
| Training Accuracy | 0.9381 |
| Test Accuracy     | 0.9308 |
| ROC-AUC           | 0.9306 |

The Random Forest achieved a training accuracy of **93.81%** and a test accuracy of **93.08%**. The difference between training and test accuracy is approximately **0.73 percentage points**, indicating a small generalization gap.

Compared with the unconstrained Decision Tree, which achieved 100% training accuracy and 89.00% test accuracy, the Random Forest demonstrates substantially better generalization. The small train-test gap suggests that averaging predictions across multiple trees successfully reduces the variance associated with a single Decision Tree.

The ROC-AUC score of **0.9306** indicates that the Random Forest has strong ability to distinguish between the two `loan_status` classes.

Top Five Important Features

| Feature                    | Importance |
| -------------------------- | ---------: |
| loan_percent_income        |   0.295855 |
| loan_grade                 |   0.176031 |
| person_income              |   0.126358 |
| person_home_ownership_RENT |   0.114634 |
| loan_int_rate              |   0.084157 |

The most important feature was `loan_percent_income`, with an importance score of approximately **0.296**. This suggests that the proportion of a person's income represented by the loan amount plays an important role in the Random Forest's predictions'.

`loan_grade`, `person_income`, rental home ownership status, and `loan_int_rate` were also among the most influential features.

Random Forest feature importance is based on the average reduction in node impurity produced by splits involving each feature across the trees in the forest. With the default Gini criterion, features that repeatedly produce large reductions in Gini impurity receive higher importance scores.

A Random Forest feature importance score differs from a Linear Regression coefficient. A regression coefficient describes the direction and magnitude of the association between a feature and the predicted numeric value. For example, a positive coefficient indicates an increase in the predicted value as the feature increases, while holding other variables constant.

Random Forest feature importance does not directly describe a positive or negative relationship. Instead, it represents how useful a feature was for creating predictive splits. Therefore, an importance score of 0.295855 does not mean that `loan_percent_income` increases the prediction by 0.295855 units.

Bagging Concept in Random Forest

Random Forest is an ensemble learning method based on bagging. During training, each Decision Tree receives a bootstrap sample of the training dataset. Bootstrap sampling selects observations randomly with replacement, meaning that some observations may appear multiple times while others may not appear in a particular tree's training sample.

In addition, at each split, the Random Forest considers a random subset of features. For classification, the default number of features considered is approximately the square root of the total number of available features.

These sources of randomness encourage individual trees to learn different patterns. The final prediction is produced by combining the predictions of multiple trees. Ensemble averaging reduces variance and makes the Random Forest less sensitive to noise or individual training observations compared with a single deep Decision Tree.

Gradient Boosting Classification

A Gradient Boosting classifier was trained using `n_estimators=100`, `learning_rate=0.1`, `max_depth=3`, and `random_state=42`.

Gradient Boosting Performance

| Metric            |  Score |
| ----------------- | -----: |
| Training Accuracy | 0.9260 |
| Test Accuracy     | 0.9269 |
| ROC-AUC           | 0.9336 |

The Gradient Boosting model achieved a training accuracy of **92.60%** and a test accuracy of **92.69%**. The extremely small difference between training and test accuracy suggests good generalization and no clear evidence of overfitting.

The model achieved a ROC-AUC score of **0.9336**, which is slightly higher than the Random Forest ROC-AUC of **0.9306**.

Random Forest achieved slightly higher test accuracy at **93.08%**, compared with **92.69% for Gradient Boosting**. However, Gradient Boosting achieved better ROC-AUC.

For this credit-risk classification problem, ROC-AUC is particularly useful because the `loan_status` classes are imbalanced. The higher ROC-AUC suggests that Gradient Boosting provides slightly better overall class-separation ability across different classification thresholds.

Gradient Boosting trains trees sequentially. Each new tree attempts to correct errors made by the existing ensemble. The `learning_rate=0.1` parameter controls the contribution of each tree, while `n_estimators=100` specifies the number of boosting stages.

Feature Ablation Study

The five features with the lowest Random Forest importance scores were removed from the training and test feature matrices. A second Random Forest classifier was then trained using identical hyperparameters and `random_state=42`.

Ablation Results

| Model                 | Number of Features |  ROC-AUC |
| --------------------- | -----------------: | -------: |
| Full Random Forest    |                 17 | 0.930578 |
| Reduced Random Forest |                 12 | 0.926395 |

The full Random Forest achieved a ROC-AUC of **0.930578**, while the reduced Random Forest achieved a ROC-AUC of **0.926395**.

The ROC-AUC decreased by approximately **0.00418**, or about **0.42 percentage points**, after removing the five lowest-importance features.

This result suggests that the removed features were not completely uninformative. Although their individual feature importance scores were low, they collectively contributed a small amount of predictive information to the Random Forest.

Therefore, the ablation experiment does not indicate that these features were simply adding noise. Removing them caused a measurable, although relatively small, reduction in class-separation performance.

Production Implications

The reduced model uses **12 features instead of 17**, representing a reduction of approximately **29% in feature dimensionality**. A lower-dimensional model may provide production advantages, including lower feature-processing requirements, reduced inference cost, and a smaller feature maintenance burden.

However, model simplification resulted in an ROC-AUC decrease of approximately **0.00418**. Whether this trade-off is acceptable depends on the production system's tolerable performance threshold.

If minimizing infrastructure and feature-maintenance complexity is a priority and an AUC degradation of approximately 0.004 is acceptable, the reduced model may be considered. If maximum predictive performance is the primary objective, the full 17-feature Random Forest should be retained.

Overall Conclusion

The Random Forest achieved strong generalization with a test accuracy of **93.08%** and ROC-AUC of **0.9306**. The small train-test accuracy gap indicates that ensemble averaging successfully reduced the high variance observed in the unconstrained Decision Tree.

Gradient Boosting achieved the highest ROC-AUC of **0.9336**, suggesting slightly stronger class-separation ability than the Random Forest.

The feature ablation study demonstrated that the five lowest-importance Random Forest features still provided a small amount of collective predictive information. Removing them reduced ROC-AUC from **0.930578 to 0.926395**.

Based on the current test-set results, Gradient Boosting is the strongest model in terms of ROC-AUC, while Random Forest provides slightly higher classification accuracy. Cross-validation should be used next to determine whether these performance differences remain consistent across multiple data splits before selecting the final production model.



In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier
)
# ============================================================
# STRATIFIED K-FOLD CROSS-VALIDATION
# ============================================================

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)
# ============================================================
# DEFINE MODELS
# ============================================================

logistic_model_cv = Pipeline([
    ("scaler", StandardScaler()),
    (
        "logistic",
        LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        )
    )
])


decision_tree_cv = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)


random_forest_cv = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42
)


gradient_boosting_cv = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)
models = {
    "Logistic Regression": logistic_model_cv,
    "Controlled Decision Tree": decision_tree_cv,
    "Random Forest": random_forest_cv,
    "Gradient Boosting": gradient_boosting_cv
}
# ============================================================
# CROSS-VALIDATED MODEL COMPARISON
# ============================================================

cv_results = []

for model_name, model in models.items():

    auc_scores = cross_val_score(
        model,
        X_train_clf,
        y_train_clf,
        cv=cv,
        scoring="roc_auc"
    )

    cv_results.append({
        "Model": model_name,
        "Mean AUC": auc_scores.mean(),
        "Std AUC": auc_scores.std()
    })

    print("=" * 60)
    print(model_name)
    print("=" * 60)

    print("Fold AUC Scores :", auc_scores)
    print(f"Mean AUC        : {auc_scores.mean():.4f}")
    print(f"Standard Dev    : {auc_scores.std():.4f}")
    print()

In [ ]:
cv_comparison = pd.DataFrame(cv_results)

cv_comparison = cv_comparison.sort_values(
    by="Mean AUC",
    ascending=False
).reset_index(drop=True)


print("=" * 70)
print("5-FOLD CROSS-VALIDATED ROC-AUC COMPARISON")
print("=" * 70)

print(cv_comparison)

In [ ]:
best_model = cv_comparison.iloc[0]

print("\n" + "=" * 60)
print("BEST CROSS-VALIDATED MODEL")
print("=" * 60)

print("Model    :", best_model["Model"])
print("Mean AUC :", round(best_model["Mean AUC"], 4))
print("Std AUC  :", round(best_model["Std AUC"], 4))

In [ ]:
Cross-Validated Model Comparison

A five-fold stratified cross-validation procedure was performed to compare four classification models: Logistic Regression, Controlled Decision Tree, Random Forest, and Gradient Boosting.

`StratifiedKFold` was configured with `n_splits=5`, `shuffle=True`, and `random_state=42`. Stratification was used to preserve approximately the same `loan_status` class distribution within each fold.

ROC-AUC was selected as the evaluation metric because the target variable is imbalanced. ROC-AUC measures a model's ability to distinguish between the two classes across different classification thresholds.

#Cross-Validation Results

| Model                    | Mean ROC-AUC | Standard Deviation |
| ------------------------ | -----------: | -----------------: |
| Logistic Regression      |       0.8638 |             0.0029 |
| Controlled Decision Tree |       0.8913 |             0.0038 |
| Random Forest            |       0.9260 |             0.0030 |
| Gradient Boosting        |       0.9280 |             0.0022 |

### Logistic Regression

Logistic Regression achieved a mean ROC-AUC of **0.8638** with a standard deviation of **0.0029**.

The relatively low standard deviation indicates that the model performs consistently across the five validation folds. However, its mean ROC-AUC is lower than the tree-based ensemble models.

This suggests that Logistic Regression may be limited by its primarily linear decision boundary and may not fully capture the complex and non-linear relationships present in the credit-risk dataset.

### Controlled Decision Tree

The Controlled Decision Tree achieved a mean ROC-AUC of **0.8913** with a standard deviation of **0.0038**.

Its performance is higher than Logistic Regression, suggesting that the Decision Tree captures non-linear relationships and feature interactions more effectively.

However, the Decision Tree has the highest standard deviation among the four models. Although the variation is still small, this indicates slightly greater sensitivity to changes in the training and validation data.

### Random Forest

Random Forest achieved a mean ROC-AUC of **0.9260** with a standard deviation of **0.0030**.

The substantial improvement over the single Decision Tree demonstrates the advantage of ensemble learning. By combining predictions from multiple Decision Trees trained on bootstrap samples and random feature subsets, Random Forest reduces the variance associated with an individual tree.

The high mean ROC-AUC indicates strong class-separation ability, while the low standard deviation demonstrates consistent performance across the five validation folds.

### Gradient Boosting

Gradient Boosting achieved the highest mean ROC-AUC of **0.9280** and the lowest standard deviation of **0.0022**.

The high mean ROC-AUC indicates that Gradient Boosting provides the strongest overall ability to distinguish between the two `loan_status` classes among the evaluated models.

In addition, the lowest standard deviation indicates that its performance is highly consistent across different validation subsets.

Gradient Boosting builds trees sequentially, with each new tree attempting to correct errors made by the existing ensemble. This sequential error-correction process allows the model to capture complex patterns and feature interactions within the credit-risk dataset.

### Why Cross-Validation Is More Reliable

A single train-test split evaluates a model using only one specific division of the dataset. The resulting performance score may be influenced by the particular observations assigned to the training and test sets.

Five-fold cross-validation evaluates each model across five different validation subsets. During each iteration, four folds are used for model training and the remaining fold is used for validation. The process is repeated until every fold has served as the validation set.

The mean ROC-AUC represents the model's average generalization performance across multiple data splits. The standard deviation measures the consistency of the model's performance between folds.

Therefore, cross-validation provides a more reliable estimate of generalization performance than a single train-test split because the model is evaluated on multiple subsets of unseen data rather than one test split alone.

### Overall Model Comparison

The cross-validation results show a clear improvement as model complexity and ensemble learning techniques are introduced.

Logistic Regression produced a mean ROC-AUC of **0.8638**, while the Controlled Decision Tree improved the score to **0.8913**. Random Forest further increased the mean ROC-AUC to **0.9260**.

Gradient Boosting achieved the highest mean ROC-AUC of **0.9280** and the lowest standard deviation of **0.0022**.

The difference between Gradient Boosting and Random Forest is relatively small at approximately **0.002 ROC-AUC points**. Therefore, the results do not indicate an extremely large performance advantage for Gradient Boosting. However, Gradient Boosting consistently achieved the strongest average performance and the lowest variation across the five folds.

### Conclusion

Based on the five-fold cross-validation results, **Gradient Boosting is the strongest model among the four evaluated classifiers**.

It achieved the highest mean ROC-AUC of **0.9280** and the lowest standard deviation of **0.0022**, indicating both strong predictive performance and consistent generalization across different validation subsets.

Random Forest remains a strong alternative with a mean ROC-AUC of **0.9260**. However, Gradient Boosting is selected as the current best-performing model based on its slightly higher average ROC-AUC and lower cross-validation variability.

The cross-validation results support the earlier test-set evaluation, where Gradient Boosting also achieved the highest ROC-AUC. Therefore, Gradient Boosting is the leading candidate for the final production pipeline, subject to the remaining model tuning and production evaluation steps.



In [ ]:
import pandas as pd

from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold


# ============================================================
# RANDOM FOREST PIPELINE
# ============================================================

rf_pipeline = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler(),
    RandomForestClassifier(random_state=42)
)


# ============================================================
# PARAMETER GRID
# ============================================================

param_grid = {
    'randomforestclassifier__n_estimators': [50, 100, 200],
    'randomforestclassifier__max_depth': [5, 10, None],
    'randomforestclassifier__min_samples_leaf': [1, 5]
}


# ============================================================
# STRATIFIED K-FOLD
# ============================================================

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


# ============================================================
# GRID SEARCH
# ============================================================

grid_search = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring='roc_auc',
    n_jobs=-1
)


# ============================================================
# TRAIN GRID SEARCH
# ============================================================

grid_search.fit(
    X_train_clf,
    y_train_clf
)


# ============================================================
# DISPLAY RESULTS
# ============================================================

print("=" * 70)
print("GRID SEARCH RESULTS")
print("=" * 70)

print("\nBest Parameters:")
print(grid_search.best_params_)

print("\nBest Cross-Validated ROC-AUC:")
print(f"{grid_search.best_score_:.4f}")


# ============================================================
# BEST PIPELINE
# ============================================================

best_rf_pipeline = grid_search.best_estimator_

print("\n" + "=" * 70)
print("BEST RANDOM FOREST PIPELINE")
print("=" * 70)

print(best_rf_pipeline)

In [ ]:
## Hyperparameter Tuning with GridSearchCV

Hyperparameter tuning was performed on the Random Forest classifier using `GridSearchCV`. The objective was to systematically identify the combination of Random Forest hyperparameters that achieved the highest cross-validated ROC-AUC score.

A Scikit-learn Pipeline was created using `SimpleImputer(strategy='median')`, `StandardScaler()`, and `RandomForestClassifier(random_state=42)`.

Using a Pipeline ensures that preprocessing operations are performed within each cross-validation fold. The imputer and scaler are fitted only on the training portion of each fold, reducing the risk of data leakage.

### Hyperparameter Search Space

The following Random Forest hyperparameters were evaluated:

* `n_estimators`: 50, 100, 200
* `max_depth`: 5, 10, None
* `min_samples_leaf`: 1, 5

The total number of hyperparameter combinations was:

3 × 3 × 2 = 18 configurations

Each configuration was evaluated using five-fold stratified cross-validation. Therefore, the total number of cross-validation model fits was:

18 × 5 = 90 model fits

### Best Hyperparameters

GridSearchCV identified the following best parameter combination:

* `max_depth = None`
* `min_samples_leaf = 1`
* `n_estimators = 200`

The best mean cross-validated ROC-AUC score was **0.9331**.

The selected `max_depth=None` allows each Decision Tree in the Random Forest to grow without a predefined depth restriction. The value `min_samples_leaf=1` allows leaf nodes to contain a minimum of one training sample.

The selected `n_estimators=200` means that the Random Forest combines predictions from 200 Decision Trees. Increasing the number of trees can improve the stability of the ensemble because predictions are averaged across a larger collection of models.

### Comparison with the Baseline Random Forest

The original Random Forest achieved a mean five-fold cross-validated ROC-AUC of **0.9260**.

After hyperparameter tuning, the best Random Forest configuration achieved a mean cross-validated ROC-AUC of **0.9331**.

The absolute improvement in ROC-AUC was:

0.9331 − 0.9260 = 0.0071

This represents a modest but measurable improvement in the model's ability to distinguish between the two `loan_status` classes.

The result indicates that the original Random Forest hyperparameters were already effective. However, GridSearchCV identified a more suitable configuration by increasing the number of trees from 100 to 200 and allowing unrestricted tree depth.

### Interpretation of the Best Configuration

The selection of `max_depth=None` suggests that deeper trees were beneficial when used as part of the Random Forest ensemble.

Although an unrestricted individual Decision Tree can have a high risk of overfitting, Random Forest reduces this risk by combining multiple trees trained using bootstrap samples and random feature subsets.

The selection of `min_samples_leaf=1` indicates that the Grid Search did not find additional leaf-size regularization beneficial within the tested parameter grid.

The selection of 200 estimators suggests that averaging predictions across a larger number of trees provided stronger cross-validated predictive performance.

### Grid Search vs Randomized Search

Grid Search exhaustively evaluates every hyperparameter combination specified in the parameter grid. This provides a systematic comparison and guarantees that every defined configuration is evaluated.

However, the computational cost of Grid Search increases rapidly as the number of hyperparameters and candidate values grows. A larger parameter grid may require hundreds or thousands of model training operations.

Randomized Search evaluates only a specified number of randomly selected hyperparameter combinations. It is generally more computationally efficient for large search spaces and allows a wider range of parameter values to be explored.

The disadvantage of Randomized Search is that it does not evaluate every possible parameter combination and may therefore miss a strong configuration.

For this project, GridSearchCV was appropriate because the parameter grid contained only 18 configurations. With five-fold cross-validation, 90 model fits were required, making exhaustive search computationally manageable.

### Conclusion

GridSearchCV successfully improved the Random Forest model's mean cross-validated ROC-AUC from **0.9260 to 0.9331**.

The best configuration used **200 trees, unrestricted tree depth, and a minimum of one sample per leaf**.

The ROC-AUC improvement of **0.0071** is relatively modest but demonstrates that systematic hyperparameter tuning can improve model performance beyond manually selected parameters.

Based on the GridSearchCV results, the tuned Random Forest Pipeline is a strong candidate for final model evaluation and production serialization.


In [ ]:
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.metrics import roc_auc_score

# ============================================================
# TRAINING FRACTIONS
# ============================================================

training_fractions = [0.2, 0.4, 0.6, 0.8, 1.0]

learning_curve_results = []

# ============================================================
# MANUAL LEARNING CURVE
# ============================================================

for fraction in training_fractions:

    # Calculate subset size
    subset_size = int(fraction * len(X_train_clf))

    # Select first fraction of training rows
    X_train_subset = X_train_clf.iloc[:subset_size]
    y_train_subset = y_train_clf.iloc[:subset_size]

    # Clone the tuned pipeline
    model = clone(best_rf_pipeline)

    # Train on subset
    model.fit(
        X_train_subset,
        y_train_subset
    )

    # Training probabilities
    train_proba = model.predict_proba(
        X_train_subset
    )[:, 1]

    # Test probabilities
    test_proba = model.predict_proba(
        X_test_clf
    )[:, 1]

    # Calculate AUC
    train_auc = roc_auc_score(
        y_train_subset,
        train_proba
    )

    test_auc = roc_auc_score(
        y_test_clf,
        test_proba
    )

    # Store results
    learning_curve_results.append({
        "Training Fraction": fraction,
        "Training AUC": train_auc,
        "Test AUC": test_auc
    })

# ============================================================
# LEARNING CURVE RESULTS
# ============================================================

learning_curve_df = pd.DataFrame(
    learning_curve_results
)

print("=" * 70)
print("MANUAL LEARNING CURVE")
print("=" * 70)

print(learning_curve_df)

In [ ]:

Manual Learning Curve Analysis

A manual learning curve was created using the best Random Forest Pipeline identified through GridSearchCV. The objective was to evaluate how the model's training and test ROC-AUC change as progressively larger portions of the training dataset are used.

The tuned Random Forest Pipeline was trained using 20%, 40%, 60%, 80%, and 100% of the available training data. For each training fraction, ROC-AUC was calculated on the corresponding training subset and on the fixed test dataset.

### Learning Curve Results

| Training Fraction | Training AUC | Test AUC |
| ----------------- | -----------: | -------: |
| 20%               |       1.0000 |   0.9181 |
| 40%               |       1.0000 |   0.9248 |
| 60%               |       1.0000 |   0.9319 |
| 80%               |       1.0000 |   0.9352 |
| 100%              |       1.0000 |   0.9389 |

### Training AUC Analysis

The training ROC-AUC remained at **1.0000 for every training fraction**. Therefore, the training AUC did not decrease as the training dataset increased.

A training AUC of 1.0 indicates that the Random Forest perfectly ranks the positive and negative classes within the training data. This behaviour is consistent with the high flexibility of the selected model configuration, particularly because GridSearchCV selected `max_depth=None` and `min_samples_leaf=1`.

These hyperparameters allow individual trees to grow deeply and create highly specific decision boundaries. Therefore, the model demonstrates signs of high training-set fit and potential overfitting.

However, perfect training performance alone does not prove that the model fails to generalize. The test ROC-AUC must also be examined.

### Test AUC Analysis

The test ROC-AUC consistently increased as the amount of training data increased.

Test AUC improved from **0.9181 at 20% of the training data** to **0.9248 at 40%** and **0.9319 at 60%**.

Performance continued to improve to **0.9352 at 80%** and reached its highest value of **0.9389 when 100% of the training data was used**.

This consistent upward trend indicates that the Random Forest benefits from additional training examples. As more observations are introduced, the model learns patterns that generalize more effectively to unseen loan applications.

### Is the Model Limited by Data Quantity or Model Capacity?

The test ROC-AUC is still increasing at the 100% training fraction. There is no clear plateau in test performance within the available training-data range.

Therefore, the current model appears to be more limited by **data quantity than model capacity**.

Collecting additional representative credit-risk observations may further improve the model's generalization performance.

Although the training AUC remains at 1.0, the test AUC also improves consistently as more training data is added. This suggests that additional data is helping reduce the generalization gap.

### Training and Test Performance Gap

At 20% of the training data, the difference between training and test AUC was approximately:

1.0000 − 0.9181 = 0.0819

At 100% of the training data, the difference decreased to approximately:

1.0000 − 0.9389 = 0.0611

The reduction in the training-test AUC gap indicates improved generalization as the amount of training data increases.

The model still demonstrates a high training fit, but the decreasing generalization gap suggests that additional data helps reduce the effect of model variance.

### Conclusion

The manual learning curve shows that training ROC-AUC remains at **1.0000 across all training fractions**, indicating that the tuned Random Forest is highly flexible and fits the training data extremely closely.

However, test ROC-AUC consistently increases from **0.9181 to 0.9389** as the training dataset grows from 20% to 100%.

Since test performance is still improving at the maximum available training fraction, the learning curve suggests that the model is currently **limited primarily by data quantity rather than model capacity**.

Therefore, collecting additional representative training data could potentially improve model generalization further. The tuned Random Forest remains a strong candidate for deployment, although its perfect training AUC should continue to be monitored as an indication of high model complexity.


In [ ]:
print("=" * 70)
print("BEST PIPELINE")
print("=" * 70)

print(best_rf_pipeline)

In [ ]:
import joblib

# ============================================================
# SERIALIZE BEST MODEL
# ============================================================

joblib.dump(
    best_rf_pipeline,
    "best_model.pkl"
)

print("Model successfully saved as best_model.pkl")

In [ ]:
import os

file_path = "best_model.pkl"

file_size_mb = os.path.getsize(file_path) / (1024 * 1024)

print("File exists :", os.path.exists(file_path))
print(f"File size   : {file_size_mb:.2f} MB")

In [ ]:
# ============================================================
# CREATE TWO HAND-CRAFTED TEST ROWS
# ============================================================

test_rows = X_train_clf.iloc[:2].copy()

# Hand-crafted applicant 1
test_rows.iloc[0, test_rows.columns.get_loc("person_age")] = 30
test_rows.iloc[0, test_rows.columns.get_loc("person_income")] = 75000
test_rows.iloc[0, test_rows.columns.get_loc("loan_amnt")] = 10000
test_rows.iloc[0, test_rows.columns.get_loc("loan_percent_income")] = 0.13

# Hand-crafted applicant 2
test_rows.iloc[1, test_rows.columns.get_loc("person_age")] = 25
test_rows.iloc[1, test_rows.columns.get_loc("person_income")] = 25000
test_rows.iloc[1, test_rows.columns.get_loc("loan_amnt")] = 20000
test_rows.iloc[1, test_rows.columns.get_loc("loan_percent_income")] = 0.80

print(test_rows)

In [ ]:
# ============================================================
# RELOAD AND PREDICT
# ============================================================

import joblib

loaded_model = joblib.load("best_model.pkl")

predictions = loaded_model.predict(test_rows)

probabilities = loaded_model.predict_proba(test_rows)[:, 1]

print("Predictions :", predictions)
print("Probabilities :", probabilities)

In [ ]:
print("=" * 70)
print("MODEL RELOAD TEST")
print("=" * 70)

print("Model loaded successfully.")
print("Number of test rows :", len(test_rows))
print("Predictions         :", predictions)

print("\nReload and prediction completed without errors.")

In [ ]:
## Model Serialization and Reload Testing

The best-performing tuned Random Forest Pipeline identified by GridSearchCV was serialized to disk using the `joblib` library.

The Pipeline contains three stages: median-value imputation using `SimpleImputer`, feature scaling using `StandardScaler`, and the tuned `RandomForestClassifier`.

The tuned Random Forest uses 200 estimators, unrestricted tree depth, and a minimum of one sample per leaf.

The complete Pipeline was saved as `best_model.pkl` using `joblib.dump()`.

### Why the Complete Pipeline Was Serialized

The complete preprocessing and modeling Pipeline was saved instead of saving only the Random Forest classifier.

This ensures that new input data passes through the same median imputation and feature-scaling transformations used during model training before predictions are generated.

Packaging preprocessing and prediction logic together reduces the risk of inconsistencies between training and production environments.

### Model Reload and Prediction Test

The serialized model was reloaded using `joblib.load('best_model.pkl')`.

Two hand-crafted applicant records were created using the same feature structure expected by the trained model. The reloaded Pipeline successfully generated class predictions and prediction probabilities for both test records without errors.

This confirms that the serialized Pipeline can be loaded in a new Python session and used for inference without retraining the model.

### Production Reproducibility

Model serialization improves reproducibility because the exact fitted preprocessing transformations and trained Random Forest parameters are preserved.

The saved `best_model.pkl` file can be loaded by a future application or inference service to generate predictions on new loan applicant data.

The successful reload-and-predict test confirms that the model artifact is suitable for integration into the later LLM-powered financial advisory system.

### Repository Storage

The serialized model file was checked for its total file size. If the file size remains below the repository's 100 MB file limit, `best_model.pkl` can be committed directly to the project repository.

If the model artifact exceeds 100 MB, the model file should not be committed directly. Instead, the training and GridSearchCV script can be included so that the model artifact can be regenerated reproducibly.

### Conclusion

The tuned Random Forest Pipeline was successfully serialized as `best_model.pkl`, reloaded using `joblib`, and tested on two hand-crafted input records.

The reloaded model generated predictions without errors, confirming that the complete preprocessing and classification workflow has been preserved successfully.

This serialized Pipeline represents the final trained machine-learning artifact and can be reused for future loan-risk prediction and financial advisory integration.


In [ ]:
from sklearn.metrics import roc_auc_score

dt_test_proba = controlled_tree.predict_proba(
    X_test_clf_scaled
)[:, 1]

dt_test_auc = roc_auc_score(
    y_test_clf,
    dt_test_proba
)

print(f"Controlled Decision Tree Test AUC: {dt_test_auc:.4f}")

In [ ]:
Summary Model Comparison

The classification models developed in Parts 2 and 3 were compared using 5-fold cross-validation ROC-AUC, cross-validation standard deviation, and test-set ROC-AUC.

| Model                    | 5-Fold CV Mean AUC | 5-Fold CV Std AUC | Test-Set AUC |
| ------------------------ | -----------------: | ----------------: | -----------: |
| Logistic Regression      |             0.8638 |            0.0029 |       0.8603 |
| Controlled Decision Tree |             0.8913 |            0.0038 |       0.8937 |
| Random Forest            |             0.9260 |            0.0030 |       0.9306 |
| Gradient Boosting        |             0.9280 |            0.0022 |       0.9336 |

Model Performance Interpretation

Logistic Regression achieved a mean cross-validation AUC of **0.8638** and a test AUC of **0.8603**. The similar cross-validation and test results indicate stable generalization, although its class-separation performance was lower than the tree-based ensemble models.

The Controlled Decision Tree improved performance with a mean cross-validation AUC of **0.8913** and a test AUC of **0.8937**. The close agreement between these values indicates that controlling tree depth reduced overfitting compared with the unconstrained Decision Tree.

Random Forest achieved a strong mean cross-validation AUC of **0.9260** and a test AUC of **0.9306**. This demonstrates that ensemble averaging substantially improved class-separation performance compared with a single Decision Tree.

Gradient Boosting produced the strongest overall results among the cross-validated baseline models, with the highest mean cross-validation AUC of **0.9280**, the lowest AUC standard deviation of **0.0022**, and the highest test AUC of **0.9336**.

Client Recommendation

I recommend the **Gradient Boosting model** for the client because it achieved the highest mean cross-validation AUC and test-set AUC among the directly compared models. Its low cross-validation standard deviation indicates stable performance across different validation folds. The model provides stronger class-separation performance than Logistic Regression, the Controlled Decision Tree, and the baseline Random Forest. Therefore, Gradient Boosting provides the best balance of predictive performance, stability, and generalization for the current loan-risk classification task.
